In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
import seaborn as sns
from scipy.stats import mannwhitneyu
from cobra.io import read_sbml_model


# Get the current working directory
current_directory = os.getcwd()
# If required go to repository root
if os.path.split(current_directory)[1] != 'PAM_Parametrization':
    # Go up two levels
    parent_directory = os.path.dirname(os.path.dirname(current_directory))
    # Change the directory to the parent directory
    os.chdir(parent_directory)
    
from PAModelpy.utils import set_up_pam
from Scripts.pam_generation import setup_ecoli_pam as set_up_ecoli_pam_curated
from Modules.PAMparametrizer.utils.sector_config_functions import get_protein_sector_config
from Modules.PAMparametrizer.utils.pamparametrizer_analysis import (get_results_from_simulations,
                                                   calculate_error_for_reactions, calculate_r_squared_for_reaction,
                                                   calculate_difference_simulation_experiment)
from Modules.PAMparametrizer.utils.pam_generation import create_pamodel_from_diagnostics_file

# from Modules.utils import calculate_r_squared_for_reaction
# from Scripts.Visualization.PAMparametrizer_progress_cleaned_figure import run_simulations

N_ALT_MODELS = 10
    
ECOLI_PHENOTYPE_DATA_PATH = os.path.join('Data', 'Ecoli_phenotypes')
MODEL_FILE_PATH = os.path.join('Models', 'iML1515.xml')

PARAM_FILE_OLD = os.path.join('Results', '1_preprocessing','proteinAllocationModel_iML1515_EnzymaticData_250912.xlsx')
PARAM_FILE_SCALED = os.path.join('Results', '2_parametrization','proteinAllocationModel_iML1515_EnzymaticData_multi.xlsx')

BEST_INDIV_RESULT_FILES = [os.path.join('Results','2_parametrization','diagnostics',
                                     f'pam_parametrizer_diagnostics_{i}.xlsx') for i in range(1,N_ALT_MODELS+1)]

# 1. Load reference data

In [ ]:
# load exchange rates for different carbon sources by Gerosa et al. (2015) in Ecoli BW25113
flux_csources = pd.read_excel(os.path.join(ECOLI_PHENOTYPE_DATA_PATH, 'Ecoli_phenotypes_py_rev.xls'),
                       sheet_name = 'Fluxes_Csources',
                            index_col=1)
flux_csources_df = flux_csources.drop(['Flux (publication)', 'Reversibility'], axis=1)
flux_csources_df.head()

# 2. Setup the *Escherichia coli* iML1515 model with new parameters

In [ ]:
#setup the model
gem = read_sbml_model(MODEL_FILE_PATH)
ecoli_pam_wt = set_up_pam(PARAM_FILE_OLD, 
                          model = MODEL_FILE_PATH, 
                          sensitivity=False) # not curation for reference
ecoli_pam_curated = set_up_ecoli_pam_curated(
    pam_data_file_path = os.path.join('Data', 'proteinAllocationModel_iML1515_EnzymaticData_py.xls'),
    sensitivity = False) # curated for reference

pam = set_up_pam(PARAM_FILE_SCALED, 
                 model = MODEL_FILE_PATH,
                 sensitivity = False)
pam.change_reaction_bounds('EX_glc__D_e', lower_bound=0)

new_ecoli_pams = {alt+1: create_pamodel_from_diagnostics_file(file,
                                          pam.copy(copy_with_pickle=True)) for alt, file in enumerate(BEST_INDIV_RESULT_FILES)}

# 3. Check internal flux distribution

## 3.1 Parse dataframes to validate the flux distribution

In [ ]:
# Get the data for growth on multiple carbon sources from Gerosa et al. (2015)
# Make sure all the fluxes of backward reactions are inverted to match the model directionality
fluxes_to_simulate = flux_csources_df.copy()
new_indices = []
for i, row in fluxes_to_simulate.iterrows():
    if isinstance(row.name, str):
        if row.name[-2:] == '_b':
            new_indices.append(row.name[:-2])
            fluxes_to_simulate.loc[row.name]= -row
        else: 
            new_indices.append(row.name)
    else:
        new_indices.append(row.name)
            
fluxes_to_simulate.index = new_indices
fluxes_to_simulate_parsed = fluxes_to_simulate[fluxes_to_simulate.index.notnull()]
fluxes_to_simulate_parsed = fluxes_to_simulate_parsed.rename(
    index = {'BIOMASS_Ec_iML1515_WT_75p37M':'BIOMASS_Ecoli_core_w_GAM'}
).drop('Glucose (flux ratio Glc)', axis = 1)

In [ ]:
# extract the validation data and substrate information for each carbon source
flux_mapper = {col: fluxes_to_simulate_parsed.index[i] for i,col in enumerate(fluxes_to_simulate_parsed.columns)}
fluxes_to_save = []
# Get the fluxes we want to save
for flux in fluxes_to_simulate_parsed.index:
    if isinstance(flux, str):
        fluxes_to_save += [f for f in flux.split(', ')]

#parse the fluxes such that we can run and validate simulations easily
validation_df = pd.DataFrame(columns = list(fluxes_to_simulate_parsed.index))
substrate_ids = []
substrate_uptake = []
for substrate, fluxes in fluxes_to_simulate_parsed.items():
    substrate_ids += [flux_mapper[substrate]]
    substrate_uptake += [fluxes.loc[flux_mapper[substrate]]]
    validation_df = pd.concat([validation_df,fluxes.to_frame().T], ignore_index =True)

validation_df.index = list(flux_mapper.values())
validation_df

## 3.2 Run simulations

In [ ]:
kwargs = {'substrate_ids': list(validation_df.index), 
          'substrate_rates': [[rate] for rate in substrate_uptake],
          'fluxes_to_save' : fluxes_to_save
         }
fluxes = {'iML1515': get_results_from_simulations(gem, **kwargs)['fluxes']}
trans_sector_config = {sub_id: get_protein_sector_config(ecoli_pam_wt,
                                                                   sub_id,
                                                                   np.arange(-4,-1,1)
                                                                  ) for sub_id in validation_df.index
                          }

unused_sector_config = {sub_id: get_protein_sector_config(ecoli_pam_wt,
                                                                   sub_id,
                                                                   np.arange(-4,-1,1),
                                                          protein_sector = 'UnusedEnzymeSector'
                                                                  ) for sub_id in validation_df.index
                          }
# for each study, run simulations
for model_label, pam in zip(['Curated', 'GotEnzymes']+list(new_ecoli_pams.keys()),
                            [ecoli_pam_curated, ecoli_pam_wt]+list(new_ecoli_pams.values())
                           ):
    trans_sector_config = {sub_id: get_protein_sector_config(pam,
                                                                   sub_id,
                                                                   np.arange(-4,-1,1)
                                                                  ) for sub_id in validation_df.index
                          }
    sector = 'UnusedEnzymeSector' if model_label!= 'Curated' else 'UnusedProteinSector'
    unused_sector_config = {sub_id: get_protein_sector_config(pam,
                                                                   sub_id,
                                                                   np.arange(-4,-1,1),
                                                              protein_sector = sector
                                                                  ) for sub_id in validation_df.index
                          }
    pam.change_reaction_bounds('EX_glc__D_e', 0, 1e3)
    fluxes[model_label] = get_results_from_simulations(pam, 
                                                       sectors_config = {
                                                           'TranslationalProteinSector': trans_sector_config, 
                                                           'UnusedEnzymeSector': unused_sector_config
                                                       },**kwargs)['fluxes']

In [ ]:
error_new_dict = {alt: calculate_error_for_reactions(validation_df,
                                                  fluxes,
                                                  fluxes_to_save) for alt, fluxes in fluxes.items()}
for alt, error_list in error_new_dict.items():
    print(f'R^2 values for alternative model {alt} with the optimized parameters: ', np.nanmean(error_list))



## 3.2 Visualize the simulation results for the different models

In [ ]:
# validation_df_1.index = validation_df.index.str.split(', ')
validation_df_1 = validation_df.T.reset_index()
validation_df_1['index'] = validation_df_1['index'].str.split(', ')
validation_df_1 = validation_df_1.explode('index').set_index('index').T
validation_df_1
# validation_df = validation_df.explode()

### Boxplots of simulation error

In [ ]:
fontsize = 15
models = [f'Alternative {alt}' for alt in error_new_dict.keys() if not isinstance(alt, str)]
model_colors = sns.color_palette("viridis", n_colors=len(models))
cmap = dict(zip(models, model_colors))
cmap = {**cmap,**{'iML1515':'white','GotEnzymes': 'grey', 'Curated': 'chocolate'}}

# Prepare the figure
fig, ax = plt.subplots(figsize=(12, 6))

# Combine data into a DataFrame
all_differences = pd.DataFrame()
curated_differences = None  # Placeholder for Curated errors
num_significant = 0

for col, (model, sub_df) in enumerate(zip(['iML1515','GotEnzymes', 'Curated']+models, fluxes.values())):
    differences = []
    for _, row in sub_df.iterrows():
        substrate_id = row['substrate_id']
        difference = calculate_difference_simulation_experiment(
            validation_df_1, row, fluxes_to_save[1:], substrate_id)
        differences += difference
    
    print(f"{model}: median = {np.median(differences)}, mean = {np.mean(differences)}, stdev = {np.std(differences)}")

    temp_df = pd.DataFrame({'Model': [model] * len(differences), 'Difference': differences})

    # Append to the main DataFrame
    all_differences = pd.concat([all_differences, temp_df], ignore_index=True)

# Boxplot or Violin Plot
sns.boxplot(x='Model', y='Difference', data=all_differences, ax=ax, palette=cmap, showfliers=False,
           )

# Set labels and title
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=fontsize)
ax.set_xlabel('Model', fontsize=fontsize * 1.5)
ax.set_ylabel(r'Difference (exp - sim) mmol/$\text{g}_{CDW}$/h', fontsize=fontsize)
ax.grid(visible=True, alpha=0.2, linewidth=0.7, axis='y')
ax.set_axisbelow(True)
plt.tight_layout()
# plt.show()
plt.savefig('Figures/SuppFig_multiple_csources_error_boxplot.png')

### line graphs of simulation per reaction

In [ ]:
# visualize per flux
fig, axs = plt.subplots(ncols = 6, nrows = 6, figsize = [25/2.56,30/2.56])
substrate_ids_cur = fluxes['Curated'].substrate_id
substrate_ids = fluxes['GotEnzymes'].substrate_id
cmap['iML1515'] = 'black'

fontsize = 12
#make the plot panels for each reaction
fig_reactions = []
for i in range(0,36,6):
    fig_reactions += [[rxn for rxn in fluxes_to_save[i:i+6]]]
# plot all reactions
for j in range(6):
    reactions_to_plot = fig_reactions[j]
    for i, rxn in enumerate(reactions_to_plot):
        validation = validation_df_1[rxn]
        axs[j,i].scatter(substrate_ids_cur.str.extract(r'^EX_(.*?)_e$')[0], validation_df_1.loc[substrate_ids_cur, rxn].abs(), color = 'grey')
#         axs[j,i].plot(substrate_ids_cur, fluxes['Curated'][rxn].abs(), label = 'curated', color = 'black')
#         axs[j,i].plot(fluxes_wt['substrate_id'], fluxes['GotEnzymes'][rxn].abs(),label = 'GotEnzymes', color = 'blue')
        for label, (model, flux_df) in enumerate(zip(['iML1515','GotEnzymes', 'Curated']+models, fluxes.values())):
            axs[j,i].plot(flux_df['substrate_id'].str.extract(r'^EX_(.*?)_e$')[0], 
                          flux_df[rxn].abs(), label = model, 
                          color = cmap[model])

#         axs[j,i].plot(substrate_ids, fluxes_new[rxn].abs(), label = 'PAMparametrizer', color = 'red')
        rxn = rxn if 'BIOMASS' not in rxn else 'Growth rate [1/h]'
        axs[j,i].set_title(rxn, fontsize=fontsize)
        axs[j,i].tick_params(axis='x', labelrotation=60)
        axs[j,i].grid(visible=True, alpha=0.2, linewidth=0.7)
        axs[j,i].set_axisbelow(True)


handles, labels = axs[j,i].get_legend_handles_labels()    
fig.legend(handles, labels, loc = 'lower center', bbox_to_anchor=(0.5, 0),ncols = 5, fontsize= fontsize)

fig.supylabel('flux rate [mmol/gCDW/h]', fontsize = fontsize*1.1,y=0.55)
fig.supxlabel('limiting nutrient', fontsize = fontsize*1.1, y=0.085)

plt.tight_layout()

plt.subplots_adjust(bottom = 0.15)
fig.savefig('Figures/SuppFig_fluxgraphs_csources')